# Continuous Feature Engineering and Weight of Evidence

This notebook carries the discrete training and test checkpoints into the original fine- and coarse-classing workflow.

## Inputs and helpers

Fine-classing tables are calculated on training data. Coarse cut points are then applied unchanged to the held-out partition.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 100)
EPSILON = 1e-6
PROCESSED = Path("../data/processed")

In [ ]:
inputs_train = pd.read_pickle(PROCESSED / "discrete_inputs_train.pkl")
inputs_test = pd.read_pickle(PROCESSED / "discrete_inputs_test.pkl")
target_train = pd.read_pickle(PROCESSED / "targets_train.pkl")
target_test = pd.read_pickle(PROCESSED / "targets_test.pkl")
inputs_train.shape, inputs_test.shape

In [ ]:
def _woe_table(inputs: pd.DataFrame, feature: str, target: pd.Series, ordered: bool) -> pd.DataFrame:
    """Build a WoE table; zero good/bad distributions are clipped to EPSILON before log."""
    frame = pd.concat([inputs[feature], target.rename("good_bad")], axis=1)
    table = frame.groupby(feature, dropna=False, observed=False)["good_bad"].agg(n_obs="count", prop_good="mean").reset_index()
    if ordered:
        table = table.sort_values(feature, key=lambda values: values.astype(str)).reset_index(drop=True)
    table["prop_n_obs"] = table["n_obs"] / table["n_obs"].sum()
    table["n_good"] = table["n_obs"] * table["prop_good"]
    table["n_bad"] = table["n_obs"] - table["n_good"]
    table["prop_n_good"] = table["n_good"] / table["n_good"].sum()
    table["prop_n_bad"] = table["n_bad"] / table["n_bad"].sum()
    good_for_log = table["prop_n_good"].clip(lower=EPSILON)
    bad_for_log = table["prop_n_bad"].clip(lower=EPSILON)
    table["WoE"] = np.log(good_for_log / bad_for_log)
    table["diff_prop_good"] = table["prop_good"].diff().abs()
    table["diff_WoE"] = table["WoE"].diff().abs()
    table["IV"] = ((good_for_log - bad_for_log) * table["WoE"]).sum()
    return table


def woe_discrete(inputs: pd.DataFrame, feature: str, target: pd.Series) -> pd.DataFrame:
    """Return category counts, good/bad distributions, WoE, and feature IV."""
    return _woe_table(inputs, feature, target, ordered=False).sort_values("WoE").reset_index(drop=True)


def woe_ordered_continuous(inputs: pd.DataFrame, feature: str, target: pd.Series) -> pd.DataFrame:
    """Return ordered-bin counts, good/bad distributions, WoE, and feature IV."""
    return _woe_table(inputs, feature, target, ordered=True)


def plot_by_woe(table: pd.DataFrame, rotation: int = 0) -> None:
    """Plot WoE in table order with readable labels."""
    plt.figure(figsize=(18, 6))
    plt.plot(table.iloc[:, 0].astype(str), table["WoE"], marker="o", linestyle="--", color="black")
    plt.xlabel(table.columns[0])
    plt.ylabel("Weight of Evidence")
    plt.title(f"Weight of Evidence by {table.columns[0]}")
    plt.xticks(rotation=rotation)
    plt.show()


def add_groups(frame: pd.DataFrame, feature: str, groups: dict[str, list[str]]) -> pd.DataFrame:
    result = frame.copy()
    values = result[feature].fillna("Missing")
    for label, members in groups.items():
        result[f"{feature}:{label}"] = values.isin(members).astype(int)
    return result

## Fine-classing procedure

The ordered helper retains the natural order of raw values or interval bins. Each fine-classing table and range-focused plot supports the original coarse grouping decision.

In [ ]:
def fine_class_table(frame: pd.DataFrame, feature: str, bins: int = 50) -> pd.DataFrame:
    fine = pd.cut(frame[feature], bins=bins)
    return woe_ordered_continuous(pd.DataFrame({f"{feature}_factor": fine}, index=frame.index), f"{feature}_factor", target_train.loc[frame.index])

## Term and employment length

In [ ]:
term_woe = woe_ordered_continuous(inputs_train, "term_int", target_train)
emp_length_woe = woe_ordered_continuous(inputs_train, "emp_length_int", target_train)
plot_by_woe(term_woe)
plot_by_woe(emp_length_woe)
display(term_woe)
display(emp_length_woe)

In [ ]:
def add_interval_groups(frame, feature, groups, output_feature=None):
    result = frame.copy()
    values = result[feature]
    for label, condition in groups.items():
        result[f"{output_feature or feature}:{label}"] = condition(values).astype(int)
    return result

term_groups = {"36": lambda x: x == 36, "60": lambda x: x == 60}
emp_length_groups = {"0": lambda x: x == 0, "1": lambda x: x == 1, "2-4": lambda x: x.between(2, 4), "5-6": lambda x: x.between(5, 6), "7-9": lambda x: x.between(7, 9), "10": lambda x: x == 10}

## Months since issue date

In [ ]:
issue_fine = fine_class_table(inputs_train, "mths_issue_date")
plot_by_woe(issue_fine, rotation=90)
plot_by_woe(issue_fine.iloc[3:], rotation=90)
display(issue_fine)

In [ ]:
issue_groups = {"<38": lambda x: x < 38, "38-39": lambda x: x.between(38, 39), "40-41": lambda x: x.between(40, 41), "42-48": lambda x: x.between(42, 48), "49-52": lambda x: x.between(49, 52), "53-64": lambda x: x.between(53, 64), "65-84": lambda x: x.between(65, 84), ">84": lambda x: x > 84}

## Interest rate and funded amount

In [ ]:
int_rate_fine = fine_class_table(inputs_train, "int_rate")
funded_amnt_fine = fine_class_table(inputs_train, "funded_amnt")
plot_by_woe(int_rate_fine, rotation=90)
plot_by_woe(funded_amnt_fine, rotation=90)
display(int_rate_fine)
display(funded_amnt_fine)

In [ ]:
int_rate_groups = {"<9.548": lambda x: x <= 9.548, "9.548-12.025": lambda x: x.gt(9.548) & x.le(12.025), "12.025-15.74": lambda x: x.gt(12.025) & x.le(15.74), "15.74-20.281": lambda x: x.gt(15.74) & x.le(20.281), ">20.281": lambda x: x > 20.281}

**Funded amount conclusion.** The original fine-classing plot is retained, but no funded-amount dummy family is added: it does not show useful monotonic discriminatory structure for this model.

## Months since earliest credit line and installment

In [ ]:
earliest_fine = fine_class_table(inputs_train, "mths_earliest_cr_line_date")
installment_fine = fine_class_table(inputs_train, "installment")
plot_by_woe(earliest_fine.iloc[20:], rotation=90)
plot_by_woe(installment_fine, rotation=90)
display(earliest_fine)
display(installment_fine)

In [ ]:
earliest_groups = {"<140": lambda x: x < 140, "141-164": lambda x: x.between(140, 164), "165-247": lambda x: x.between(165, 247), "248-270": lambda x: x.between(248, 270), "271-352": lambda x: x.between(271, 352), ">352": lambda x: x > 352}

## Delinquencies, inquiries, open accounts, and public records

In [ ]:
ordered_count_features = ["delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec"]
count_woe = {feature: woe_ordered_continuous(inputs_train, feature, target_train) for feature in ordered_count_features}
for feature, table in count_woe.items():
    display(table)
    plot_by_woe(table, rotation=90)
plot_by_woe(count_woe["delinq_2yrs"].iloc[:-12])
plot_by_woe(count_woe["open_acc"].iloc[:40], rotation=90)

In [ ]:
delinq_groups = {"0": lambda x: x == 0, "1-3": lambda x: x.between(1, 3), ">=4": lambda x: x >= 4}
inq_groups = {"0": lambda x: x == 0, "1-2": lambda x: x.between(1, 2), "3-6": lambda x: x.between(3, 6), ">6": lambda x: x > 6}
open_acc_groups = {"0": lambda x: x == 0, "1-3": lambda x: x.between(1, 3), "4-12": lambda x: x.between(4, 12), "13-17": lambda x: x.between(13, 17), "18-22": lambda x: x.between(18, 22), "23-25": lambda x: x.between(23, 25), "26-30": lambda x: x.between(26, 30), ">=31": lambda x: x >= 31}
pub_rec_groups = {"0-2": lambda x: x.between(0, 2), "3-4": lambda x: x.between(3, 4), ">=5": lambda x: x >= 5}

The original notebook labelled the last delinquency band `>=4` but tested `>=9`. The condition is corrected to `>=4`, matching its label and the stated grouping.

## Total accounts and accounts currently delinquent

In [ ]:
total_acc_fine = fine_class_table(inputs_train, "total_acc")
acc_now_delinq_woe = woe_ordered_continuous(inputs_train, "acc_now_delinq", target_train)
plot_by_woe(total_acc_fine, rotation=90)
plot_by_woe(acc_now_delinq_woe)
display(total_acc_fine)
display(acc_now_delinq_woe)

In [ ]:
total_acc_groups = {"<=27": lambda x: x <= 27, "28-51": lambda x: x.between(28, 51), ">=52": lambda x: x >= 52}
acc_now_delinq_groups = {"0": lambda x: x == 0, ">=1": lambda x: x >= 1}

## Revolving high credit limit, installment, and income

In [ ]:
rev_limit_fine = fine_class_table(inputs_train, "total_rev_hi_lim", bins=2000)
annual_inc_fine = fine_class_table(inputs_train, "annual_inc")
plot_by_woe(rev_limit_fine.iloc[:50], rotation=90)
plot_by_woe(annual_inc_fine, rotation=90)
display(rev_limit_fine)
display(annual_inc_fine)

In [ ]:
rev_limit_groups = {"<=5K": lambda x: x <= 5000, "5K-10K": lambda x: x.gt(5000) & x.le(10000), "10K-20K": lambda x: x.gt(10000) & x.le(20000), "20K-30K": lambda x: x.gt(20000) & x.le(30000), "30K-40K": lambda x: x.gt(30000) & x.le(40000), "40K-55K": lambda x: x.gt(40000) & x.le(55000), "55K-95K": lambda x: x.gt(55000) & x.le(95000), ">95K": lambda x: x > 95000}
annual_inc_groups = {"<20K": lambda x: x <= 20000, "20K-30K": lambda x: x.gt(20000) & x.le(30000), "30K-40K": lambda x: x.gt(30000) & x.le(40000), "40K-50K": lambda x: x.gt(40000) & x.le(50000), "50K-60K": lambda x: x.gt(50000) & x.le(60000), "60K-70K": lambda x: x.gt(60000) & x.le(70000), "70K-80K": lambda x: x.gt(70000) & x.le(80000), "80K-90K": lambda x: x.gt(80000) & x.le(90000), "90K-100K": lambda x: x.gt(90000) & x.le(100000), "100K-120K": lambda x: x.gt(100000) & x.le(120000), "120K-140K": lambda x: x.gt(120000) & x.le(140000), ">140K": lambda x: x > 140000}

## Missingness-aware variables: recent delinquency, DTI, and public record

In [ ]:
last_delinq_nonmissing = inputs_train.loc[inputs_train["mths_since_last_delinq"].notna()]
last_record_nonmissing = inputs_train.loc[inputs_train["mths_since_last_record"].notna()]
last_delinq_fine = fine_class_table(last_delinq_nonmissing, "mths_since_last_delinq")
last_record_fine = fine_class_table(last_record_nonmissing, "mths_since_last_record")
dti_fine = fine_class_table(inputs_train, "dti", bins=100)
dti_under_35 = fine_class_table(inputs_train.loc[inputs_train["dti"] <= 35], "dti")
for table in [last_delinq_fine, last_record_fine, dti_fine, dti_under_35]:
    display(table)
    plot_by_woe(table, rotation=90)

In [ ]:
last_delinq_groups = {"Missing": lambda x: x.isna(), "0-3": lambda x: x.between(0, 3), "4-30": lambda x: x.between(4, 30), "31-56": lambda x: x.between(31, 56), ">=57": lambda x: x >= 57}
dti_groups = {"<=1.4": lambda x: x <= 1.4, "1.4-3.5": lambda x: x.gt(1.4) & x.le(3.5), "3.5-7.7": lambda x: x.gt(3.5) & x.le(7.7), "7.7-10.5": lambda x: x.gt(7.7) & x.le(10.5), "10.5-16.1": lambda x: x.gt(10.5) & x.le(16.1), "16.1-20.3": lambda x: x.gt(16.1) & x.le(20.3), "20.3-21.7": lambda x: x.gt(20.3) & x.le(21.7), "21.7-22.4": lambda x: x.gt(21.7) & x.le(22.4), "22.4-35": lambda x: x.gt(22.4) & x.le(35), ">35": lambda x: x > 35}
last_record_groups = {"Missing": lambda x: x.isna(), "0-2": lambda x: x.between(0, 2), "3-20": lambda x: x.between(3, 20), "21-31": lambda x: x.between(21, 31), "32-80": lambda x: x.between(32, 80), "81-86": lambda x: x.between(81, 86), ">86": lambda x: x > 86}

## Apply coarse classes and save final inputs

Missing-value categories are explicit where absence is informative. All boundaries below are copied from the training investigation and never re-estimated on test data.

In [ ]:
continuous_groups = {"term": term_groups, "emp_length": emp_length_groups, "mths_issue_date": issue_groups, "int_rate": int_rate_groups, "mths_earliest_cr_line_date": earliest_groups, "delinq_2yrs": delinq_groups, "inq_last_6mths": inq_groups, "open_acc": open_acc_groups, "pub_rec": pub_rec_groups, "total_acc": total_acc_groups, "acc_now_delinq": acc_now_delinq_groups, "total_rev_hi_lim": rev_limit_groups, "annual_inc": annual_inc_groups, "mths_since_last_delinq": last_delinq_groups, "dti": dti_groups, "mths_since_last_record": last_record_groups}
source_columns = {"term": "term_int", "emp_length": "emp_length_int", **{key: key for key in continuous_groups if key not in {"term", "emp_length"}}}
model_inputs_train = inputs_train.copy()
model_inputs_test = inputs_test.copy()
for output_feature, groups in continuous_groups.items():
    source_feature = source_columns[output_feature]
    model_inputs_train = add_interval_groups(model_inputs_train, source_feature, groups, output_feature)
    model_inputs_test = add_interval_groups(model_inputs_test, source_feature, groups, output_feature)

model_inputs_train.to_pickle(PROCESSED / "model_inputs_train.pkl")
model_inputs_test.to_pickle(PROCESSED / "model_inputs_test.pkl")
print("Saved model checkpoints:", model_inputs_train.shape, model_inputs_test.shape)

## Final feature inventory and conclusions

The final matrix includes the original discrete families plus coarse classes for term, employment length, issue date, interest rate, credit history, account behavior, revolving limit, income, delinquency recency, DTI, and public-record recency. Funded amount and installment are investigated and plotted; funded amount is deliberately excluded because its WoE does not supply useful monotonic discrimination. Reference categories are selected later with the logistic specification.